# PostgreSQL with Python and Psycopg

**Goal:** create and manage a small database from Python, execute SQL safely, and understand transactions and constraints.

This notebook creates a separate practice database called `books_data`; it does **not** use the `imdb` database.

## 1. Set up the connection

Psycopg sends SQL statements to PostgreSQL. Edit the connection settings for your installation. Install the driver once if needed.

In [ ]:
# If needed, install once:
# %pip install "psycopg[binary]"

import psycopg
from psycopg.errors import DuplicateDatabase


DB_NAME = "books_data"
DB_USER = "postgres"
DB_PASSWORD = ""  # Enter your password if required.
DB_HOST = "localhost"
DB_PORT = 5432

## 2. Create a practice database

To create a database, connect to an existing database first (here, `postgres`). `CREATE DATABASE` must run outside a transaction, so use `autocommit=True`.

**Key Terms**
- *Cursor*: designed for navigating through and interacting with records in a database.
- *Transaction*: a coherent unit of work within a database, encapsulating a series of operations into a single, indivisible process.

If the database already exists, the cell leaves it in place. The next section deliberately recreates the example tables, so do not use this notebook with a database containing data you need to keep.

In [ ]:
admin_connection = psycopg.connect(
    dbname="postgres",
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    autocommit=True
)

In [ ]:
try:
    with admin_connection.cursor() as cursor:
        cursor.execute(f"CREATE DATABASE {DB_NAME}")

    print(f"Database {DB_NAME!r} created.")
except DuplicateDatabase:
    print(f"Database {DB_NAME!r} already exists; reusing it.")
finally:
    admin_connection.close()

## 3. Create tables and define constraints

A **primary key** uniquely identifies each row and cannot be `NULL`. `NOT NULL`, `UNIQUE`, and `CHECK` constraints validate data. A **foreign key** links rows across tables.

Here, one author can write many books (a one-to-many relationship). `ON DELETE RESTRICT` prevents deleting an author who still has books.

In [ ]:
connection = psycopg.connect(
    dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT
)

cursor = connection.cursor()

# Reset only these tables.
cursor.execute("DROP TABLE IF EXISTS books")
cursor.execute("DROP TABLE IF EXISTS authors")
cursor.execute(
    """CREATE TABLE authors (
    id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    nationality VARCHAR(2) CHECK (nationality SIMILAR TO '[A-Z]{2}')
)
"""
)

cursor.execute(
    """CREATE TABLE books (
    id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    author_id INTEGER NOT NULL REFERENCES authors(id) ON DELETE RESTRICT,
    publication_year INTEGER CHECK (publication_year > 0)
)
"""
)

Changes initially belong to the current transaction: `commit()` (or `connection.autocommit = True` as global setting) makes them permanent.

In [ ]:
connection.commit()

## 4. Insert rows safely

Use `%s` placeholders for values instead of building SQL with string concatenation. `execute()` runs a statement once; `executemany()` runs it for multiple sets of values.

The SQL statement and the values are passed separately. This is safer and handles quoting correctly.

In [ ]:
cursor.executemany(
    "INSERT INTO authors (id, name, nationality) VALUES (%s, %s, %s)",
    [
        (1, "Isaac Asimov", "US"),
        (2, "George Orwell", "UK"),
        (3, "Ursula K. Le Guin", "US")
    ]
)

cursor.executemany(
    "INSERT INTO books (id, title, author_id, publication_year) VALUES (%s, %s, %s, %s)",
    [
        (1, "Foundation", 1, 1951),
        (2, "1984", 2, 1949),
        (3, "A Wizard of Earthsea", 3, 1968),
        (4, "I, Robot", 1, 1950)
    ]
)

connection.commit()

## 5. Read query results

After `execute()` runs a `SELECT` query, `fetchall()`, `fetchone()`, and `fetchmany(n)` read all rows, one row, or up to `n` rows, respectively. Each row is returned as a tuple.

SQL filtering and sorting happen in PostgreSQL, before the rows are sent to Python.

In [ ]:
cursor.execute(
    """SELECT books.id, books.title, authors.name AS author, books.publication_year
FROM books
INNER JOIN authors ON books.author_id = authors.id
WHERE publication_year >= %s
ORDER BY publication_year DESC, title ASC
""",
    params=(1950,)
)

In [ ]:
# for row in cursor.fetchall():
#     print(row)

print(cursor.fetchone())

In [ ]:
cursor.execute("SELECT title FROM books WHERE id = %s", params=(2,))
print("Book with id 2:", cursor.fetchone())

## 6. Update and delete rows

`UPDATE` changes existing rows and `DELETE` removes rows. Always check the `WHERE` clause: without it, every row is affected. Use parameters for values here, too.

In [ ]:
cursor.execute("UPDATE books SET publication_year = %s WHERE id = %s", params=(1951, 1))
print("Updated rows:", cursor.rowcount)

In [ ]:
cursor.execute("DELETE FROM books WHERE id = %s", params=(4,))
print("Deleted rows:", cursor.rowcount)
connection.commit()

In [ ]:
cursor.execute("SELECT * FROM books")
for row in cursor.fetchall():
    print(row)

In [ ]:
cursor.execute("INSERT INTO books VALUES (4, 'I, Robot', 1, 1950)")
print("Inserted rows:", cursor.rowcount)
connection.commit()

## 7. Transactions: `commit()` and `rollback()`

Changes are part of a transaction until they are committed. `commit()` makes them permanent; `rollback()` discards uncommitted changes.

Run this cell to change a title and then undo that change. The final query confirms that the original title remains.

In [ ]:
cursor.execute(
    "UPDATE books SET title = %s WHERE id = %s", params=("Temporary title", 2)
)

connection.rollback()

In [ ]:
cursor.execute("SELECT title FROM books WHERE id = %s", params=(2,))
print("Title after rollback:", cursor.fetchone()[0])

## 8. Constraints and referential integrity

Try the examples one at a time. They intentionally violate constraints:

- a duplicate primary key;
- a reference to an author that does not exist;
- deleting an author who still has books.

PostgreSQL rejects each invalid operation. After a failed statement, the current transaction must be rolled back before it can continue.

In [ ]:
# Uncomment and run ONE statement at a time to observe the constraint error:
# cursor.execute("INSERT INTO authors (id, name) VALUES (%s, %s)", (1, "Another author"))
# cursor.execute("INSERT INTO books (id, title, author_id) VALUES (%s, %s, %s)", (10, "Unknown author", 999))
# cursor.execute("DELETE FROM authors WHERE id = %s", (1,))

In [ ]:
# If a statement fails, recover the connection with:
connection.rollback()

## 9. Exercises

Complete these tasks before opening the instructor solutions below.

1. Insert a new author with ID `4`, name `"Franz Kafka"` and Czech (`"CZ"`) nationality.
2. Add the books `"Das Schloß"` (ID `5`, author ID `4`, publication year `1926`) and `"The Castle"` (ID `6`, author ID `4`, publication year `1930`).
3. Retrieve all books published before 1960, showing the title and author, sorted by year.
4. Change the publication year of book `5` to `1980`, but then undo the change with `rollback()`. Verify the stored year.
5. Try changing author `4` nationality to `"cz"`, `"CZE"` and `"AT"` (Austria). See which works and explain why.

## Solutions

In [ ]:
# Exercise 1
cursor.execute(
    "INSERT INTO authors (id, name, nationality) VALUES (%s, %s, %s)",
    params=(4, "Franz Kafka", "CZ")
)

# Exercise 2
cursor.executemany(
    "INSERT INTO books (id, title, author_id, publication_year) VALUES (%s, %s, %s, %s)",
    [(5, "Das Schloß", 4, 1926), (6, "The Castle", 4, 1930)]
)

connection.commit()

In [ ]:
# Exercise 3
cursor.execute(
    """SELECT books.title, authors.name, books.publication_year
FROM books
INNER JOIN authors ON books.author_id = authors.id
WHERE books.publication_year < %s
ORDER BY books.publication_year, books.title
""",
    params=(1960,)
)

for row in cursor.fetchall():
    print(row)

In [ ]:
# Exercise 4
cursor.execute(
    "UPDATE books SET publication_year = %s WHERE id = %s",
    params=(1980, 5),
)

connection.rollback()
cursor.execute("SELECT publication_year FROM books WHERE id = %s", params=(5,))
print(cursor.fetchone()[0])  # 1926

In [ ]:
# Exercise 5
cursor.execute(
    "UPDATE authors SET nationality = %s WHERE id = %s",
    params=("AT", 4),  # "CZE", "AT"
)

In [ ]:
# connection.rollback()
connection.commit()
cursor.execute("SELECT * FROM authors")
for row in cursor.fetchall():
    print(row)

## 11. Close resources

Close the cursor and connection when the database work is complete

In [ ]:
cursor.close()
connection.close()

### Workflow to remember

**Connect → create tables and constraints → execute parameterized SQL → fetch results → commit or roll back → close resources.**